In [34]:
# !pip install -q \
# transformers \
# sentence-transformers \
# faiss-cpu \
# wikipedia \
# streamlit \
# pyngrok \
# pandas \
# numpy \
# tqdm

In [36]:
%%writefile app.py

import numpy as np
import wikipedia
import faiss
import streamlit as st

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

# =========================
# CONFIGURATION
# =========================

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

GENERATION_MODEL = "google/flan-t5-small"

CHUNK_SIZE = 100
OVERLAP = 30
TOP_K = 5

# =========================
# LOAD MODELS
# =========================

@st.cache_resource
def load_models():

    embed_model = SentenceTransformer(
        EMBEDDING_MODEL
    )

    tokenizer = AutoTokenizer.from_pretrained(
        GENERATION_MODEL
    )

    model_gen = AutoModelForSeq2SeqLM.from_pretrained(
        GENERATION_MODEL
    )

    return embed_model, tokenizer, model_gen


embed_model, tokenizer, model_gen = load_models()

# =========================
# WIKIPEDIA RETRIEVAL
# =========================

def retrieve_wikipedia_context(query):

    documents = []

    try:

        page = wikipedia.page(
            query,
            auto_suggest=True
        )

        documents.append(page.content)

    except Exception:

        try:

            results = wikipedia.search(
                query,
                results=3
            )

            for title in results:

                try:

                    page = wikipedia.page(title)

                    documents.append(
                        page.content
                    )

                except:
                    continue

        except:
            pass

    return "\n".join(documents)

# =========================
# CHUNKING
# =========================

def chunk_text(text):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + CHUNK_SIZE

        chunk = words[start:end]

        chunks.append(
            " ".join(chunk)
        )

        start += (
            CHUNK_SIZE - OVERLAP
        )

    return chunks

# =========================
# BUILD FAISS INDEX
# =========================

def build_faiss_index(chunks):

    embeddings = embed_model.encode(
        chunks,
        normalize_embeddings=True
    )

    embeddings = np.array(
        embeddings
    ).astype("float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(
        dimension
    )

    index.add(embeddings)

    return index

# =========================
# RETRIEVAL
# =========================

def retrieve_chunks(question, chunks, index):

    query_embedding = embed_model.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        TOP_K
    )

    retrieved = []

    for idx in indices[0]:

        retrieved.append(
            chunks[idx]
        )

    return retrieved

# =========================
# PROMPT
# =========================

def build_prompt(context, question):

    return f"""
Answer ONLY using the provided context.

If answer is missing,
say:
I don't know.

Context:
{context}

Question:
{question}

Answer:
"""

# =========================
# GENERATION
# =========================

def generate_answer(question):

    context = retrieve_wikipedia_context(
        question
    )

    if len(context.strip()) == 0:

        return (
            "I don't know.",
            ""
        )

    chunks = chunk_text(context)

    index = build_faiss_index(chunks)

    retrieved_chunks = retrieve_chunks(
        question,
        chunks,
        index
    )

    final_context = "\n\n".join(
        retrieved_chunks
    )

    prompt = build_prompt(
        final_context,
        question
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model_gen.generate(
        **inputs,
        max_new_tokens=32
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, final_context

# =========================
# STREAMLIT UI
# =========================

st.set_page_config(
    page_title="RAG QA System",
    layout="wide"
)

st.title(
    "Retrieval-Augmented Generation (RAG) QA System"
)

st.write(
    "Ask any factual question."
)

question = st.text_input(
    "Enter your question:"
)

if st.button("Generate Answer"):

    if question.strip() == "":

        st.warning(
            "Please enter a question."
        )

    else:

        with st.spinner(
            "Generating answer..."
        ):

            answer, retrieved_context = generate_answer(
                question
            )

        st.subheader(
            "Generated Answer"
        )

        st.success(answer)

        st.subheader(
            "Retrieved Context"
        )

        st.write(retrieved_context)

Overwriting app.py


In [37]:
!streamlit run app.py &>/content/logs.txt &

In [38]:
from pyngrok import ngrok

ngrok.set_auth_token("3DeyAqyddpdOfMBE0paUqLNXU69_3oRstBQQhoKw3qtNtU9rE")

In [39]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://preachy-mustard-muck.ngrok-free.dev" -> "http://localhost:8501"
